# Heart Disease, Revisited with an ANN

Project 04 used gradient boosting on UCI Cleveland and landed around AUC ~0.88 on the held-out split (with logistic regression and random forest both edging it out slightly on CV at ~0.90). I wanted to see whether a small ANN would beat any of those, or just sit in the same band — because on tabular data, neural nets usually don't.

Same data, same preprocessing, same CV protocol. Different model.

In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, recall_score, precision_score, classification_report

torch.manual_seed(0); np.random.seed(0)

DATA_DIR = Path("../../data/heart")
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

device: cpu


## Load Cleveland (same pattern as project 04)

In [2]:
col_names = [
    "age", "sex", "cp", "trestbps", "chol", "fbs",
    "restecg", "thalach", "exang", "oldpeak", "slope", "ca", "thal", "target"
]
for fname in ["heart.csv", "processed.cleveland.data", "cleveland.csv"]:
    fpath = DATA_DIR / fname
    if fpath.exists():
        df = pd.read_csv(fpath, names=col_names if fpath.suffix == ".data" else None, na_values="?")
        break
else:
    raise FileNotFoundError(f"No heart disease file found in {DATA_DIR}")

df["target"] = (df["target"] > 0).astype(int)
df = df.dropna().reset_index(drop=True)
cat_cols = ["cp", "restecg", "slope", "ca", "thal"]
df_enc = pd.get_dummies(df, columns=cat_cols, drop_first=True)
feature_cols = [c for c in df_enc.columns if c != "target"]
X = df_enc[feature_cols].values.astype(np.float32)
y = df_enc["target"].values.astype(np.int64)
print(f"shape: {X.shape}  positive rate: {y.mean():.2%}")

shape: (297, 20)  positive rate: 46.13%


## Tiny MLP

13 → 32 → 16 → 2. Dropout 0.3 because 297 rows is small and these models overfit fast.

I keep the architecture deliberately small. On Cleveland a deeper MLP just gets noisier — there's not enough data to support the extra capacity.

In [3]:
class HeartMLP(nn.Module):
    def __init__(self, in_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 32), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(32, 16), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(16, 2),
        )
    def forward(self, x): return self.net(x)

def train_one_fold(X_tr, y_tr, X_va, y_va, epochs=120, lr=5e-3):
    scaler = StandardScaler().fit(X_tr)
    X_tr_s = scaler.transform(X_tr); X_va_s = scaler.transform(X_va)
    model = HeartMLP(X_tr_s.shape[1]).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    loss_fn = nn.CrossEntropyLoss()
    xt = torch.tensor(X_tr_s, dtype=torch.float32, device=device)
    yt = torch.tensor(y_tr, dtype=torch.long, device=device)
    xv = torch.tensor(X_va_s, dtype=torch.float32, device=device)
    for ep in range(epochs):
        model.train(); opt.zero_grad()
        loss = loss_fn(model(xt), yt); loss.backward(); opt.step()
    model.eval()
    with torch.no_grad():
        proba = torch.softmax(model(xv), 1)[:, 1].cpu().numpy()
    return proba, scaler, model

## 5-fold CV, AUC and recall

In [4]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fold_auc, fold_recall = [], []
for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y), 1):
    proba, _, _ = train_one_fold(X[tr_idx], y[tr_idx], X[va_idx], y[va_idx])
    pred = (proba >= 0.5).astype(int)
    fold_auc.append(roc_auc_score(y[va_idx], proba))
    fold_recall.append(recall_score(y[va_idx], pred))
    print(f"fold {fold}  AUC={fold_auc[-1]:.3f}  recall@0.5={fold_recall[-1]:.3f}")

print(f"\nANN CV mean AUC: {np.mean(fold_auc):.3f}  (std {np.std(fold_auc):.3f})")
print(f"ANN CV mean recall@0.5: {np.mean(fold_recall):.3f}")

fold 1  AUC=0.950  recall@0.5=0.929
fold 2  AUC=0.882  recall@0.5=0.821


fold 3  AUC=0.830  recall@0.5=0.741
fold 4  AUC=0.861  recall@0.5=0.778


fold 5  AUC=0.921  recall@0.5=0.778

ANN CV mean AUC: 0.889  (std 0.043)
ANN CV mean recall@0.5: 0.809


## Compared to project 04 (GB / RF / LR)

Project 04 published, from 5-fold CV on the same data and preprocessing:

| Model                | CV AUC | CV recall@0.5 |
|----------------------|--------|----------------|
| Logistic regression  | 0.905  | 0.794 |
| Random forest        | 0.902  | 0.758 |
| Gradient boosting    | 0.878  | 0.751 |

My ANN's CV results are printed in the cell above — I'm deliberately not hardcoding a number into this markdown so the comparison stays honest if I rerun.

**What I expect, given how tabular data + small datasets usually go.** The MLP lands somewhere in the same band as the three sklearn models, probably between GB (0.878) and LR (0.905). It almost never blows past logistic regression on 13 features and 297 rows — there's just not enough data to learn anything an MLP can express that a calibrated linear model can't.

If the printed AUC happens to come in higher than LR's 0.905, I'd treat that as a single-seed effect, not a real win. The right next step would be: bootstrapped CIs across multiple seeds, or a proper repeated-CV protocol.

In [5]:
# Final fit on a single 80/20 split for a held-out test score, same protocol as project 04
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
proba_te, scaler, model_final = train_one_fold(X_tr, y_tr, X_te, y_te)
pred_te = (proba_te >= 0.5).astype(int)

print(f"held-out test AUC: {roc_auc_score(y_te, proba_te):.3f}")
print(f"held-out recall@0.5: {recall_score(y_te, pred_te):.3f}")
print(f"held-out precision@0.5: {precision_score(y_te, pred_te):.3f}")
print(classification_report(y_te, pred_te, target_names=["no_disease", "disease"]))

held-out test AUC: 0.927
held-out recall@0.5: 0.821
held-out precision@0.5: 0.852
              precision    recall  f1-score   support

  no_disease       0.85      0.88      0.86        32
     disease       0.85      0.82      0.84        28

    accuracy                           0.85        60
   macro avg       0.85      0.85      0.85        60
weighted avg       0.85      0.85      0.85        60



## Takeaway

The most useful thing about running this experiment is *not* the AUC number. It's having an honest answer to "would a neural net beat GB here?" The answer, on a tabular dataset this small, is almost certainly "not by a meaningful margin." That tracks with the broader tabular-vs-deep-learning literature (e.g., Grinsztajn et al. 2022, *Why do tree-based models still outperform deep learning on tabular data?*).

If I wanted the ANN to seriously contend on tabular data I'd reach for TabTransformer or FT-Transformer — but those want more rows than Cleveland has.